In [18]:
import pandas as pd
import time
from google_play_scraper import Sort, reviews

# Configuration for Doctor On Demand
app_id = 'com.doctorondemand.android.patient'
language = 'en'
country = 'us'
review_count = 50        # Reviews to fetch per request
desired_total_reviews = 923  # Stop after this many reviews

all_reviews = []

# Initial fetch for 4-star reviews
result, continuation_token = reviews(
    app_id,
    lang=language,
    country=country,
    sort=Sort.MOST_RELEVANT,
    count=review_count,
    filter_score_with=4   # ✅ restrict to 5-star reviews
)
all_reviews.extend(result)

# Continue fetching more 4-star reviews
while continuation_token and len(all_reviews) < desired_total_reviews:
    more_reviews, continuation_token = reviews(
        app_id,
        lang=language,
        country=country,
        sort=Sort.MOST_RELEVANT,
        count=review_count,
        continuation_token=continuation_token,
        filter_score_with=4   # ✅ keep filtering by 5-star
    )
    all_reviews.extend(more_reviews)
    print(f"Fetched {len(all_reviews)} reviews so far...")
    time.sleep(1)  # small pause to avoid throttling

# Build DataFrame with review text, rating, and helpful votes
df = pd.DataFrame(all_reviews)
df_5star = df[['content', 'score', 'thumbsUpCount']]
df_5star.rename(columns={
    "content": "Review_Text",
    "score": "Rating",
    "thumbsUpCount": "Helpful_Votes"
}, inplace=True)

# Export to Excel
excel_file = 'DoctorOnDemand_4-1star_reviews.xlsx'
df_5star.to_excel(excel_file, index=False)
print(f"✅ Exported 3-star reviews with helpful votes to {excel_file}")

Fetched 100 reviews so far...
Fetched 150 reviews so far...
Fetched 200 reviews so far...
Fetched 250 reviews so far...
Fetched 300 reviews so far...
Fetched 350 reviews so far...
Fetched 400 reviews so far...
Fetched 450 reviews so far...
Fetched 500 reviews so far...
Fetched 550 reviews so far...
Fetched 600 reviews so far...
Fetched 650 reviews so far...
Fetched 700 reviews so far...
Fetched 750 reviews so far...
Fetched 800 reviews so far...
Fetched 850 reviews so far...
Fetched 900 reviews so far...
Fetched 923 reviews so far...


C:\Users\bbb4464\AppData\Local\Temp\ipykernel_29896\213301350.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_5star.rename(columns={


✅ Exported 3-star reviews with helpful votes to DoctorOnDemand_4-1star_reviews.xlsx


In [4]:
import pandas as pd
import time
from google_play_scraper import Sort, reviews

# === Config: STC Bank (Google Play) ===
app_id = "sa.com.stcbank"   # package id from the URL
language = "en"             # e.g., "en" or "ar"
country = "sa"              # Saudi Arabia store
sort_order = Sort.MOST_RELEVANT

# Pagination / limits
review_count = 2           # reviews per request (max ≈ 200)
desired_total_reviews = 248  # stop after this many (or when no more pages)

# Star filter: choose 1..5 or set to None for all
filter_stars = 2

# Output
excel_file = f"stc_bank_{language}_{country}_{filter_stars}star_reviews.xlsx"

all_reviews = []

# ----- Initial fetch -----
result, continuation_token = reviews(
    app_id,
    lang=language,
    country=country,
    sort=sort_order,
    count=review_count,
    filter_score_with=filter_stars  # restrict to selected star rating
)
all_reviews.extend(result)
print(f"Fetched {len(all_reviews)} reviews so far...")

# ----- Continue fetching until limit or no more pages -----
while continuation_token and len(all_reviews) < desired_total_reviews:
    more_reviews, continuation_token = reviews(
        app_id,
        lang=language,
        country=country,
        sort=sort_order,
        count=min(review_count, desired_total_reviews - len(all_reviews)),
        continuation_token=continuation_token,
        filter_score_with=filter_stars
    )
    all_reviews.extend(more_reviews)
    print(f"Fetched {len(all_reviews)} reviews so far...")
    time.sleep(1)  # be polite

# ----- Build DataFrame -----
df = pd.DataFrame(all_reviews)

# Keep common fields; add safe copies to avoid SettingWithCopy warnings
cols = ["content", "score", "thumbsUpCount", "at", "reviewId", "userName", "reviewCreatedVersion"]
available = [c for c in cols if c in df.columns]
df_out = df[available].copy()

df_out = df_out.rename(columns={
    "content": "Review_Text",
    "score": "Rating",
    "thumbsUpCount": "Helpful_Votes",
    "at": "Review_Time_UTC",
    "reviewCreatedVersion": "App_Version"
})

# Make timestamps Excel-friendly (ISO 8601 strings)
if "Review_Time_UTC" in df_out.columns:
    df_out["Review_Time_UTC"] = pd.to_datetime(df_out["Review_Time_UTC"], errors="coerce", utc=True)\
                                   .dt.strftime("%Y-%m-%d %H:%M:%S")

# ----- Export to Excel -----
# Requires: pip install openpyxl
df_out.to_excel(excel_file, index=False)
print(f"✅ Exported {len(df_out)} {filter_stars}-star reviews to {excel_file}")


Fetched 2 reviews so far...
Fetched 4 reviews so far...
Fetched 6 reviews so far...
Fetched 8 reviews so far...
Fetched 10 reviews so far...
Fetched 12 reviews so far...
Fetched 14 reviews so far...
Fetched 16 reviews so far...
Fetched 18 reviews so far...
Fetched 20 reviews so far...
Fetched 22 reviews so far...
Fetched 24 reviews so far...
Fetched 26 reviews so far...
Fetched 28 reviews so far...
Fetched 30 reviews so far...
Fetched 32 reviews so far...
Fetched 34 reviews so far...
Fetched 36 reviews so far...
Fetched 38 reviews so far...
Fetched 40 reviews so far...
Fetched 42 reviews so far...
Fetched 44 reviews so far...
Fetched 46 reviews so far...
Fetched 48 reviews so far...
Fetched 50 reviews so far...
Fetched 52 reviews so far...
Fetched 54 reviews so far...
Fetched 56 reviews so far...
Fetched 58 reviews so far...
Fetched 60 reviews so far...
Fetched 62 reviews so far...
Fetched 64 reviews so far...
Fetched 66 reviews so far...
Fetched 68 reviews so far...
Fetched 70 reviews

In [6]:
import pandas as pd
import time
from google_play_scraper import Sort, reviews

# === Config: STC Bank (Google Play) ===
app_id = "com.ubercab.eats"   # package id from the URL
language = "en"             # e.g., "en" or "ar"
country = "us"              # Saudi Arabia store
sort_order = Sort.MOST_RELEVANT

# Pagination / limits
review_count = 50           # reviews per request (max ≈ 200)
desired_total_reviews = 20000  # stop after this many (or when no more pages)

# Star filter: choose 1..5 or set to None for all
filter_stars = 5

# Output
excel_file = f"stc_bank_{language}_{country}_{filter_stars}star_reviews.xlsx"

all_reviews = []

# ----- Initial fetch -----
result, continuation_token = reviews(
    app_id,
    lang=language,
    country=country,
    sort=sort_order,
    count=review_count,
    filter_score_with=filter_stars  # restrict to selected star rating
)
all_reviews.extend(result)
print(f"Fetched {len(all_reviews)} reviews so far...")

# ----- Continue fetching until limit or no more pages -----
while continuation_token and len(all_reviews) < desired_total_reviews:
    more_reviews, continuation_token = reviews(
        app_id,
        lang=language,
        country=country,
        sort=sort_order,
        count=min(review_count, desired_total_reviews - len(all_reviews)),
        continuation_token=continuation_token,
        filter_score_with=filter_stars
    )
    all_reviews.extend(more_reviews)
    print(f"Fetched {len(all_reviews)} reviews so far...")
    time.sleep(1)  # be polite

# ----- Build DataFrame -----
df = pd.DataFrame(all_reviews)

# Keep common fields; add safe copies to avoid SettingWithCopy warnings
cols = ["content", "score", "thumbsUpCount", "at", "reviewId", "userName", "reviewCreatedVersion"]
available = [c for c in cols if c in df.columns]
df_out = df[available].copy()

df_out = df_out.rename(columns={
    "content": "Review_Text",
    "score": "Rating",
    "thumbsUpCount": "Helpful_Votes",
    "at": "Review_Time_UTC",
    "reviewCreatedVersion": "App_Version"
})

# Make timestamps Excel-friendly (ISO 8601 strings)
if "Review_Time_UTC" in df_out.columns:
    df_out["Review_Time_UTC"] = pd.to_datetime(df_out["Review_Time_UTC"], errors="coerce", utc=True)\
                                   .dt.strftime("%Y-%m-%d %H:%M:%S")

# ----- Export to Excel -----
# Requires: pip install openpyxl
df_out.to_excel(excel_file, index=False)
print(f"✅ Exported {len(df_out)} {filter_stars}-star reviews to {excel_file}")


Fetched 50 reviews so far...
Fetched 100 reviews so far...
Fetched 150 reviews so far...
Fetched 200 reviews so far...
Fetched 250 reviews so far...
Fetched 300 reviews so far...
Fetched 350 reviews so far...
Fetched 400 reviews so far...
Fetched 450 reviews so far...
Fetched 500 reviews so far...
Fetched 550 reviews so far...
Fetched 600 reviews so far...
Fetched 650 reviews so far...
Fetched 700 reviews so far...
Fetched 750 reviews so far...
Fetched 800 reviews so far...
Fetched 850 reviews so far...
Fetched 900 reviews so far...
Fetched 950 reviews so far...
Fetched 1000 reviews so far...
Fetched 1050 reviews so far...
Fetched 1100 reviews so far...
Fetched 1150 reviews so far...
Fetched 1200 reviews so far...
Fetched 1250 reviews so far...
Fetched 1300 reviews so far...
Fetched 1350 reviews so far...
Fetched 1400 reviews so far...
Fetched 1450 reviews so far...
Fetched 1500 reviews so far...
Fetched 1550 reviews so far...
Fetched 1600 reviews so far...
Fetched 1650 reviews so far.